# Namma Space — Phase 5: 3D Gaussian Splatting Training (Colab / Kaggle)

Trains a metric, Y-up 3D Gaussian Splat (`splatfacto`) from the aligned dataset bundle (`train_pack.zip`).

### **Colab Runtime Selection (Required)**
1. Go to menu: **Runtime** → **Change runtime type**.
2. Select **Hardware accelerator**: **T4 GPU** (free tier) or **A100 GPU** / **V100 GPU** (Colab Pro).
3. Set **Runtime architecture**: **Python 3**.
4. Click **Save**.

### Key Requirements:
- **Metric Y-up Coordinate Preservation**: Anti-centering, anti-orientation, and anti-scaling flags guarantee that the model trains directly in the metric room frame (floor at $y = 0.00\text{ m}$).
- **Presets**: `fast` (7,000 steps smoke test) or `final` (30,000 steps, SH degree 3).
- **Evaluation**: 10% held-out test frames evaluated for PSNR, SSIM, and LPIPS into `outputs/metrics.json`.
- **Export**: Exports trained Gaussian Splat to `outputs/scene_raw.ply` and verifies frame consistency.

## 1. Environment & GPU Audit

In [ ]:
# Check assigned GPU and driver
!nvidia-smi

import torch
print(f"PyTorch version   : {torch.__version__}")
print(f"CUDA available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device        : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total        : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    raise SystemError("No GPU detected! Go to Runtime > Change runtime type and select T4 GPU.")

## 2. Install Pinned Dependencies (Nerfstudio, gsplat & LPIPS)

In [ ]:
# Install ninja, torchvision, gsplat, lpips, open3d, and nerfstudio with pinned versions
!pip install --upgrade pip
!pip install ninja==1.11.1.4 jaxtyping==0.2.25 open3d==0.18.0 lpips==0.1.4 scikit-image==0.24.0
!pip install gsplat==1.0.0 --extra-index-url https://docs.gsplat.studio/whl/pt21cu118
!pip install nerfstudio==1.1.5 --no-deps
# Install necessary runtime dependencies for nerfstudio
!pip install rich tyro trimesh plyfile mediapy av xatlas h5py nuscenes-devkit pymeshlab

# Verify ns-train and ns-export
!ns-train --help > /dev/null && echo "[OK] ns-train installed successfully!"
!ns-export --help > /dev/null && echo "[OK] ns-export installed successfully!"

## 3. Upload & Unpack Training Bundle (`train_pack.zip`)

Upload the `outputs/home/train_pack.zip` file generated locally by `scripts/pack_dataset.py`.

In [ ]:
import os, zipfile, shutil
from pathlib import Path

zip_name = "train_pack.zip"
if not os.path.exists(zip_name):
    try:
        from google.colab import files
        print("Please select and upload train_pack.zip from your machine:")
        uploaded = files.upload()
    except ImportError:
        print("Running outside Colab. Ensure train_pack.zip is in current working directory.")

# Extract bundle to dataset directory
os.makedirs("dataset", exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall("dataset")

print("Extracted files:")
!ls -lh dataset
print(f"Images count: {len(os.listdir('dataset/images'))}")

## 4. Train 3D Gaussian Splat (`splatfacto`)

Choose between:
- `"fast"` : 7,000 steps (~5 min on T4)
- `"final"`: 30,000 steps, SH degree 3 (~25 min on T4)

> **CRITICAL**: The flags below disable orientation PCA, auto-centering, and pose auto-scaling, guaranteeing that the model trains directly in the metric room frame.

In [ ]:
# Select preset: 'fast' (7k smoke test) or 'final' (30k full quality)
PRESET = "fast"   # Set to "final" for production run

if PRESET == "fast":
    max_iters = 7000
    extra_flags = "--pipeline.model.num-downscales 2"
else:
    max_iters = 30000
    extra_flags = "--pipeline.model.sh-degree 3"

# Execute ns-train with anti-centering and anti-scaling flags
!ns-train splatfacto \
  --data dataset \
  --output-dir outputs/nerfstudio_models \
  --experiment-name home \
  --pipeline.datamanager.dataparser.orientation-method none \
  --pipeline.datamanager.dataparser.center-method none \
  --pipeline.datamanager.dataparser.auto-scale-poses False \
  --pipeline.datamanager.dataparser.scale-factor 1.0 \
  --pipeline.datamanager.dataparser.eval-mode fraction \
  --pipeline.datamanager.dataparser.eval-ratio 0.1 \
  --max-num-iterations {max_iters} \
  {extra_flags}

## 5. Export to `outputs/scene_raw.ply`

In [ ]:
import glob, shutil, os

cfg_candidates = sorted(glob.glob("outputs/nerfstudio_models/home/splatfacto/*/config.yml"), key=os.path.getmtime)
assert len(cfg_candidates) > 0, "No training config.yml found!"
latest_cfg = cfg_candidates[-1]
print(f"Latest training config: {latest_cfg}")

os.makedirs("outputs", exist_ok=True)

# Export Gaussian Splat PLY
!ns-export gaussian-splat --load-config {latest_cfg} --output-dir outputs/

# Rename exported PLY to scene_raw.ply
for name in ["splat.ply", "gaussian_splat.ply", "model.ply"]:
    p = os.path.join("outputs", name)
    if os.path.exists(p):
        shutil.move(p, "outputs/scene_raw.ply")
        print("Successfully saved: outputs/scene_raw.ply")
        break

!ls -lh outputs/scene_raw.ply

## 6. Evaluate Held-Out Frames (10% Holdout: PSNR, SSIM, LPIPS)

In [ ]:
# Compute PSNR, SSIM, and LPIPS metrics on held-out test frames using ns-eval
!ns-eval --load-config {latest_cfg} --output-path outputs/metrics.json

import json
if os.path.exists("outputs/metrics.json"):
    with open("outputs/metrics.json", "r") as f:
        metrics = json.load(f)
    print("\n--- Test Set Metrics ---")
    print(json.dumps(metrics.get("results", metrics), indent=2))

## 7. Verify Metric Y-up Coordinate Frame

In [ ]:
import numpy as np
import open3d as o3d

# Load splat point cloud to verify floor level and extents
pcd = o3d.io.read_point_cloud("outputs/scene_raw.ply")
pts = np.asarray(pcd.points)

y_p2 = np.percentile(pts[:, 1], 2.0)
x_span = np.percentile(pts[:, 0], 98.0) - np.percentile(pts[:, 0], 2.0)
z_span = np.percentile(pts[:, 2], 98.0) - np.percentile(pts[:, 2], 2.0)

print(f"Gaussians Count     : {len(pts):,}")
print(f"Floor Elevation (Y) : {y_p2:+.3f} m  (PASS if |Y| <= 0.15 m)")
print(f"Room Extents (X x Z): {x_span:.2f} m x {z_span:.2f} m")

assert abs(y_p2) <= 0.15, f"Floor elevation {y_p2:.3f} m deviates from y = 0.00 m!"
print("\n[PASS] Splat coordinate frame is strictly metric Y-up with floor at y = 0.00 m!")

## 8. Download Output Artifacts

In [ ]:
try:
    from google.colab import files
    print("Downloading scene_raw.ply...")
    files.download("outputs/scene_raw.ply")
    print("Downloading metrics.json...")
    files.download("outputs/metrics.json")
except ImportError:
    print("Artifacts ready in outputs/scene_raw.ply and outputs/metrics.json")